# **DSK807: Applied Machine Learning**
## Exercise: Credit Card Default Prediction

In this exercise, you will work with a credit-card default dataset and compare several machine-learning approaches studied in the course.

**Main question:**  
> Which model would you use to predict credit-card default, and what evidence supports your choice?

## 1. Dataset

The dataset contains information about **30,000 credit-card customers**.

For each customer, information is available about:

- credit limit,
- demographic characteristics,
- payment status over the previous six months,
- monthly bill amounts,
- previous payment amounts.

The **target variable** indicates whether the customer defaulted on their credit-card payment in the following month.

Your first task is to understand the dataset before building any model.

More information about the dataset is available here: https://archive.ics.uci.edu/dataset/350/defaultofcreditcardclients

## 2. Load the Dataset

Load the dataset provided for this exercise.

Inspect the first observations, dimensions, column names, and data types.

In [1]:
# Load the dataset here
import pandas as pd

df = pd.read_csv('credit_card_clients.csv') # File is available on itslearning
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [2]:
df.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
       'default payment next month'],
      dtype='str')

## 3. Prepare the Data

Prepare `X` and `y`.

- Remove columns that should not be used as predictors.
- Create a stratified train/test split.
- Keep the test set unchanged until final evaluation.


In [3]:
from sklearn.model_selection import train_test_split

# TODO: define the target column
target = "default payment next month"

X = df.drop(columns=[target])
y = df[target]

# TODO: remove ID if it is present
if "ID" in X.columns:
    X = X.drop(columns=["ID"])

# One-hot encode categorical variables
categorical_cols = ["SEX", "EDUCATION", "MARRIAGE"]

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=False,
    dtype=int
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))


(24000, 33) (6000, 33)
default payment next month
0    0.778792
1    0.221208
Name: proportion, dtype: float64


## 4. Baseline

Create a simple baseline classifier.

Evaluate it using:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

**Question:** Is accuracy alone sufficient for this dataset?


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

# TODO:
# 1. Train a Logistic Regression baseline.
# 2. Generate class predictions and probabilities.
# 3. Calculate the five evaluation metrics.


## 5. Random Forest

Train a Random Forest classifier.

Try a few different settings for:

- `n_estimators`
- `max_depth`
- `max_features`

Record the results and select your best configuration.

**Discuss:** Does increasing the number/depth of trees always improve test performance?


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TODO: train, evaluate, and experiment with different parameters


## 6. AdaBoost

Train an AdaBoost classifier.

Experiment with combinations of:

- `n_estimators`
- `learning_rate`

Suggested values:

`n_estimators = [50, 100, 200]`

`learning_rate = [0.01, 0.1, 1.0]`

**Discuss:** What happens when the learning rate changes?


In [ ]:
from sklearn.ensemble import AdaBoostClassifier

# TODO: Train, evaluate, and try several combinations


## 7. Gradient Boosting

Train a Gradient Boosting classifier.

Experiment with:

- `n_estimators`
- `learning_rate`
- `max_depth`

**Discuss:** Can a smaller learning rate be compensated for by using more trees?


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# TODO: train, evaluate, and try other settings


## 8. XGBoost — Grid Search

Now use **GridSearchCV** to search systematically for a good XGBoost configuration.

Search over:

- number of trees,
- learning rate,
- tree depth.

Use **F1-score** as the optimization metric.


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV


param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 0.3],
    "max_depth": [2, 4, 6]
}


### Inspect the Grid Search

Look at the best configurations.

**Discuss:**

1. What parameter combination performed best?
2. Is the best configuration much better than the next-best configurations?
3. What relationship do you observe between `learning_rate` and `n_estimators`?
4. Does a deeper tree always perform better?


In [ ]:
results = pd.DataFrame(grid.cv_results_)

results[[
    "param_n_estimators",
    "param_learning_rate",
    "param_max_depth",
    "mean_test_score",
    "std_test_score"
]].sort_values("mean_test_score", ascending=False).head(10)


### Evaluate the Best XGBoost Model

Evaluate `grid.best_estimator_` on the untouched test set.

Then try changing the Grid Search scoring metric from `f1` to either:

- `accuracy`
- `roc_auc`

**Question:** Do you obtain the same best parameters?


In [ ]:
best_xgb = grid.best_estimator_

# TODO: evaluate best_xgb on the test set


## 9. Neural Network

For the neural network, first scale the input features.

Keep the **same training and test observations** used by the previous models.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### Build an Experimental Neural Network

Create a function that allows you to change:

- number of hidden layers,
- number of neurons per layer.

Keep the other settings fixed initially.


In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense

tf.random.set_seed(42)

def build_model(hidden_layers=3, neurons=32):
    model = Sequential()
    model.add(Input(shape=(X_train_scaled.shape[1],)))

    # TODO: add the requested number of hidden layers
    # Each hidden layer should use ReLU.

    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


## 10. Neural Network Architecture Experiment

Test several architectures while keeping the training settings the same.

| Experiment | Hidden Layers | Neurons per Layer |
|---|---:|---:|
| A | 3 | 16 |
| B | 3 | 64 |
| C | 4 | 64 |
| D | 5 | 64 |
| E | 5 | 128 |

For each model, record training, validation, and test performance.

**Discuss:**

- Does adding layers always improve performance?
- Does increasing the number of neurons always help?
- Which network appears to overfit the most?


In [ ]:
# Example
model = build_model(hidden_layers=3, neurons=16)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    verbose=0
)

# TODO:
# Repeat for the other architectures and record the results.


### Plot the Learning Curves

Plot training and validation loss for selected architectures.

What evidence of overfitting can you see?


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


## 11. Final Comparison

Complete a final comparison of your models.

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | | | | | |
| Random Forest | | | | | |
| AdaBoost | | | | | |
| Gradient Boosting | | | | | |
| XGBoost | | | | | |
| Best Neural Network | | | | | |

### Final Discussion

1. Which model would you choose for this problem?
2. Did hyperparameter tuning make a meaningful difference?
3. Was the largest neural network the best?
4. Which model gives the best balance between performance and complexity?
5. Would your model choice change if detecting defaults (recall) were more important than overall accuracy?
